In [1]:
import pysam
import numpy as np
import pandas as pd
import itertools
import re

from Bio import SeqIO
from Bio.Seq import Seq
from tqdm import tqdm
from collections import defaultdict

# import re
# import matplotlib.pyplot as plt
# import seaborn as sns
# import scipy.stats as stats
# import os
# import mpl_scatter_density
# import statsmodels.api as sm
# from collections import Counter
# from scipy.stats import gaussian_kde, iqr
# from collections import defaultdict
# from patsy import dmatrices

In [2]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"

ref_fasta = working_path + "/data/ref/GRCh38_full_analysis_set_plus_decoy_hla.fa"
bam_file  = working_path + "/data/RRBS/SRR6294859.sorted.bam"

In [3]:
cut_str   = 'C|CGG'

In [4]:
ref_dict  = SeqIO.to_dict(SeqIO.parse(ref_fasta, "fasta"))

# parse read pairs

In [5]:
def get_align_flag(align, min_mapping_quality =30):
    is_paired  = align.is_proper_pair
    is_mapped  = (not align.is_unmapped)
    is_primary = (not align.is_secondary) and (not align.is_supplementary)
    is_qualified = (not align.is_qcfail) and align.mapping_quality >= min_mapping_quality
    is_fully_align = all(op in (0, 7, 8) for op, length in align.cigar)
    return is_paired and is_mapped and is_primary and is_qualified and is_fully_align


def read_pair_generator(bam_file, region_string=None):
    """
    Generate read pairs in a BAM file or within a region string.
    Reads are added to read_dict until a pair is found.
    """
    read_dict = defaultdict(lambda: [None, None])
    bam = pysam.AlignmentFile(bam_file, "rb")
    #bam.reset()
    
    for align in bam.fetch(until_eof=True, region=region_string):        
        if not get_align_flag(align):
            continue
        
        qname = align.query_name
        if qname not in read_dict.keys():
            read_dict[qname][int(align.is_read2)] = align
        else:
            read_dict[qname][int(align.is_read2)] = align
            yield read_dict[qname]
            del read_dict[qname]

                    
def gen_frag_df(bam_file, contig_id):
    read_pair_dict = {}
    
    for read1, read2 in read_pair_generator(bam_file, contig_id):
        pos_list   = [read1.reference_start, read1.reference_end, read2.reference_start, read2.reference_end]
        start, end = np.min(pos_list), np.max(pos_list)
        if (contig_id, start, end) in read_pair_dict:
            read_pair_dict[(contig_id, start, end)] += 1
        else:
            read_pair_dict[(contig_id, start, end)] = 1
    
    return pd.DataFrame([(*key, val) for key, val in read_pair_dict.items()])

In [ ]:
chr_list = list(ref_dict.keys())[0:25]
frag_df_list = []

for contig_id in chr_list:
    frag_df_list.append(gen_frag_df(bam_file, contig_id))

In [ ]:
frag_df =  pd.concat(frag_df_list, axis=0, ignore_index=True)
frag_df.columns = ['chr', 'start', 'end', 'count']

In [ ]:
frag_df.loc[1:10,:]

# check and assign cut site

In [ ]:
def check_boundary(contig_id, pos, site_list):
    offset  = max([len(site) for site in site_list])
    pos_l   = int(pos) - offset
    pos_r   = int(pos) + offset
    tmp_seq = ref_dict[contig_id][pos_l:pos_r].seq.upper()
    
    return sum([tmp_seq.count(site) for site in site_list])

In [ ]:
def find_closest_cut(contig_id, pos, site_list, is_upstream = True, search_window = 500):
    offset  = max([len(site) for site in site_list])
    
    if is_upstream:
        pos_l = int(pos) - search_window
        pos_r = int(pos) + offset
        tmp_seq = ref_dict[contig_id][pos_l:pos_r].seq.upper()
        
        pos_list= []
        for ix, site in enumerate(site_list):
            pos_list.extend([match.start() for match in re.finditer(site, str(tmp_seq))])
        
        pos_arr = np.array(pos_list) - search_window
        cut_pos = np.nan if len(pos_arr)==0 else pos + pos_arr[np.argmin(np.abs(pos_arr))]
        return cut_pos
        
    else:
        pos_l = int(pos) - offset
        pos_r = int(pos) + search_window
        tmp_seq = ref_dict[contig_id][pos_l:pos_r].seq.upper()
        
        pos_list= []
        for ix, site in enumerate(site_list):
            pos_list.extend([match.start() + len(site) for match in re.finditer(site, str(tmp_seq))])
        
        pos_arr = np.array(pos_list) - offset
        cut_pos = np.nan if len(pos_arr)==0 else pos + pos_arr[np.argmin(np.abs(pos_arr))]
        return cut_pos

In [ ]:
site_list = ['CCGG']

frag_df.loc[:, "flag_start"]= frag_df.apply(lambda x: check_boundary(x.chr, x.start, site_list), axis=1)
frag_df.loc[:, "flag_end"]  = frag_df.apply(lambda x: check_boundary(x.chr, x.end, site_list), axis=1)
frag_df.loc[:, "ref_l"] = frag_df.apply(lambda x: find_closest_cut(x.chr, x.start, site_list, is_upstream=True), axis=1)
frag_df.loc[:, "ref_r"] = frag_df.apply(lambda x: find_closest_cut(x.chr, x.end,  site_list, is_upstream=False), axis=1)

In [ ]:
pd.set_option("display.max_rows", 20)

In [ ]:
frag_df

In [ ]:
pd.set_option("display.max_rows", None)

In [ ]:
frag_df.loc[0:100,:]

In [ ]:
pd.set_option("display.max_rows", 20)

# calculate length, count, cg ratio, #cuts

In [ ]:
{site: 0 for site in site_list}

In [ ]:
def cal_gc_ratio(contig_id, pos_l, pos_r):
    tmp_seq = ref_dict[contig_id][int(pos_l):int(pos_r)].seq.upper()
    cg_count= tmp_seq.count("C") + tmp_seq.count("G")
    return cg_count/(pos_r - pos_l)

def cal_num_cuts(contig_id, pos_l, pos_r, site_list):
    tmp_seq  = ref_dict[contig_id][int(pos_l):int(pos_r)].seq.upper()
    site_dict= {site: 0 for site in site_list}
    for site in site_dict.keys():
        site_dict[site] = tmp_seq.count(site)
    return list(site_dict.values())

In [ ]:
keep_idx = np.logical_not(np.isnan(frag_df.ref_l) | np.isnan(frag_df.ref_r))
frag_cut_df = frag_df.loc[keep_idx, :].groupby(['chr', 'ref_l', 'ref_r'])['count'].sum().reset_index()
frag_cut_df.loc[:, "length"]  = frag_cut_df.ref_r - frag_cut_df.ref_l
frag_cut_df.loc[:, "gc_ratio"]= frag_cut_df.apply(lambda x: cal_gc_ratio(x.chr, x.ref_l, x.ref_r), axis=1)
site_num_df = frag_cut_df.apply(lambda x: cal_num_cuts(x.chr, x.ref_l, x.ref_r, site_list), axis=1, result_type='expand')
site_num_df.columns = site_list
frag_cut_df = pd.concat([frag_cut_df, site_num_df], axis=1)
frag_cut_df.loc[:, "log_count"] = np.log2(frag_cut_df.loc[:, "count"]+1)

In [ ]:
from scipy.stats import gaussian_kde, iqr
max_coverage = np.quantile(frag_cut_df['count'], 0.75) + 1.5*iqr(frag_cut_df['count'])
min_coverage = np.quantile(frag_cut_df['count'], 0.25) - 1.5*iqr(frag_cut_df['count'])

max_log_coverage = np.quantile(frag_cut_df['log_count'], 0.75) + 1.5*iqr(frag_cut_df['log_count'])

In [ ]:
frag_cut_df.loc[:, 'count'].describe()

In [ ]:
frag_cut_df.loc[:, 'log_count'].describe()

In [ ]:
max_coverage

In [ ]:
min_coverage

In [ ]:
max_log_coverage

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
 
# Creating dataset
fig = plt.figure(figsize =(10, 7))

# Creating plot
plt.boxplot(frag_cut_df.loc[:, 'count'])
# show plot
plt.show()


# Creating dataset
fig = plt.figure(figsize =(10, 7))
# Creating plot
plt.boxplot(frag_cut_df.loc[:, 'log_count'])
# show plot
plt.show()

# on the count

In [ ]:
frag_cut_df2 = frag_cut_df.loc[frag_cut_df.loc[:,"count"] < max_coverage,:]
frag_cut_df2.loc[:, "relative_depth"] = frag_cut_df2.loc[:, "count"]/max_coverage

In [ ]:
frag_cut_df2

In [ ]:
frag_cut_df2.loc[:, 'count'].describe()

In [ ]:
frag_cut_df2.gc_ratio.describe()

In [ ]:
frag_cut_df2.gc_ratio.hist()

In [ ]:
frag_cut_df2.relative_depth.describe()

In [ ]:
frag_cut_df2.relative_depth.hist()

In [ ]:
frag_cut_df2.CCGG.describe()

In [ ]:
frag_cut_df2.CCGG.hist()

In [ ]:
frag_cut_df2.length.describe()

In [ ]:
frag_cut_df2.length.hist()

In [ ]:
frag_cut_df2.loc[:, 'count'].hist(bins=100)

In [ ]:
import seaborn as sns
%matplotlib inline

corr = frag_cut_df2.loc[frag_cut_df2.loc[:,"count"]>0,["length","gc_ratio", "CCGG", "relative_depth"]].corr(method='spearman')
mask = np.triu(np.ones_like(corr, dtype=bool))

# plot the heatmap
sns.heatmap(corr,  vmin=-1, vmax=1, annot=True, cmap='BrBG', mask=mask,
            xticklabels=corr.columns,
            yticklabels=corr.columns)

In [ ]:
frag_cut_df2

In [ ]:
frag_cut_df2.to_csv("/home/wbguo/iproject/BSReadSim/test/data/RRBS/frag_cut_df2.csv", 
                    index = None, header=True, quoting=None)

# linear regression

In [ ]:
from patsy import dmatrices
import statsmodels.api as sm

In [ ]:
y, X = dmatrices('relative_depth ~ gc_ratio + length + CCGG', 
                 data = frag_cut_df2,
                 return_type='dataframe')

In [ ]:
mod = sm.OLS(y, X)
mod = mod.fit()
print(mod.summary())

In [ ]:
X

In [ ]:
y

In [ ]:
plt.scatter(y, mod.predict(X))
plt.show()

In [ ]:
plt.hist2d(np.squeeze(y.to_numpy()), mod.predict(X), cmap=plt.cm.jet)
plt.show()

# quantile regression

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [ ]:
mod = smf.quantreg('relative_depth ~ CCGG + length + gc_ratio', frag_cut_df2)
res = mod.fit(q=.5)
print(res.summary())

In [ ]:
plt.scatter(y, res.predict(X))
plt.show()

In [ ]:
plt.hist2d(np.squeeze(y.to_numpy()), res.predict(X), cmap=plt.cm.jet)
plt.show()

# GLM regression

In [ ]:
binom_glm = sm.GLM(y, X, family=sm.families.Binomial())
binom_reg = binom_glm.fit()

plt.plot(y, binom_reg.predict(X), 'o', alpha=0.2);

In [ ]:
plt.hist2d(np.squeeze(y.to_numpy()), binom_reg.predict(X), cmap=plt.cm.jet)
plt.show()

In [ ]:
# libraries
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kde

In [ ]:
y.shape

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(r2_score(y, binom_reg.predict(X)))
print(mean_squared_error(y, binom_reg.predict(X)))
print(mean_absolute_error(y, binom_reg.predict(X)))

In [ ]:
# Evaluate a gaussian kde on a regular grid of nbins x nbins over data extents
nbins=100
y = np.squeeze(y)
y_pred = np.array(binom_reg.predict(X))

k = kde.gaussian_kde([y, y_pred])
xi, yi = np.mgrid[y.min():y.max():nbins*1j, y_pred.min():y_pred.max():nbins*1j]
zi = k(np.vstack([xi.flatten(), yi.flatten()]))

# Make the plot
plt.pcolormesh(xi, yi, zi.reshape(xi.shape), shading='auto')
plt.show()

# multivariate Spline regression

In [ ]:
from pyearth import Earth

model = Earth(max_degree =3)
model.fit(X,y)

In [ ]:
plt.scatter(y, model.predict(X))
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(r2_score(y, model.predict(X)))
print(mean_squared_error(y, model.predict(X)))
print(mean_absolute_error(y, model.predict(X)))

In [ ]:
# Evaluate a gaussian kde on a regular grid of nbins x nbins over data extents
nbins=100
y = np.squeeze(y)
y_pred = np.array(model.predict(X))

k = kde.gaussian_kde([y, y_pred])
xi, yi = np.mgrid[y.min():y.max():nbins*1j, y_pred.min():y_pred.max():nbins*1j]
zi = k(np.vstack([xi.flatten(), yi.flatten()]))
 
# Make the plot
plt.pcolormesh(xi, yi, zi.reshape(xi.shape), shading='auto')
plt.show()

# beta regression

In [ ]:
from scipy.special import loggamma
from scipy.special import expit, logit

def logLikelihood(params, y, X):
    b = np.array(params[0:-1])      # the beta parameters of the regression model
    phi = params[-1]                # the phi parameter
    mu = expit(np.dot(X,b))
   
    eps = 1e-6                      # used for safety of the gamma and log functions avoiding inf
    res = - np.sum(loggamma(phi+eps) # the log likelihood
                   - loggamma(mu*phi+eps) 
                   - loggamma((1-mu)*phi+eps) 
                   + (mu*phi-1)*np.log(y+eps) 
                   + ((1-mu)*phi-1)*np.log(1-y+eps))

    return res

In [ ]:
from scipy.optimize import minimize

# initial parameters for optimization
phi = 1
b0 = 1
x0 = np.array([b0,b0,b0,b0,phi])

res = minimize(logLikelihood, x0=x0, args=(y,X), bounds=[(None,None), 
                                                         (None,None), 
                                                         (None,None), 
                                                         (None,None), 
                                                         (None,None), 
                                                         (0,None)])

# log count

In [ ]:
frag_cut_df3 = frag_cut_df.loc[frag_cut_df.loc[:,"log_count"] < max_log_coverage,:]
frag_cut_df3.loc[:, "relative_depth"] = frag_cut_df3.loc[:, "log_count"]/max_log_coverage

In [ ]:
frag_cut_df3

In [ ]:
frag_cut_df3.loc[:, 'log_count'].describe()

In [ ]:
frag_cut_df3.gc_ratio.describe()

In [ ]:
frag_cut_df3.gc_ratio.hist()

In [ ]:
frag_cut_df3.relative_depth.describe()

In [ ]:
frag_cut_df3.relative_depth.hist()

In [ ]:
frag_cut_df3.CCGG.describe()

In [ ]:
frag_cut_df3.CCGG.hist()

In [ ]:
frag_cut_df3.length.describe()

In [ ]:
frag_cut_df3.length.hist()

In [ ]:
frag_cut_df3.loc[:, 'log_count'].hist(bins=100)

In [ ]:
import seaborn as sns
%matplotlib inline

corr = frag_cut_df3.loc[frag_cut_df3.loc[:,"log_count"]>0,["length","gc_ratio", "CCGG", "relative_depth"]].corr(method='spearman')
mask = np.triu(np.ones_like(corr, dtype=bool))

# plot the heatmap
sns.heatmap(corr,  vmin=-1, vmax=1, annot=True, cmap='BrBG', mask=mask,
            xticklabels=corr.columns,
            yticklabels=corr.columns)

In [ ]:
y, X = dmatrices('relative_depth ~ gc_ratio + length + CCGG', 
                 data = frag_cut_df3,
                 return_type='dataframe')

mod = sm.OLS(y, X)
mod = mod.fit()
print(mod.summary())

In [ ]:
plt.scatter(y, mod.predict(X))
plt.show()

In [ ]:
plt.hist2d(np.squeeze(y.to_numpy()), mod.predict(X), cmap=plt.cm.jet)
plt.show()

In [ ]:
mod = smf.quantreg('relative_depth ~ CCGG + length + gc_ratio', frag_cut_df3)
res = mod.fit(q=.5)
print(res.summary())

In [ ]:
plt.scatter(y, res.predict(X))
plt.show()

# neural network

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

np.random.seed(0)
rand_idx = np.random.choice(frag_cut_df2.index, size=1000, replace=False)

# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(frag_cut_df2.loc[rand_idx, ['gc_ratio', 'length', 'CCGG']], 
                                                    frag_cut_df2.loc[rand_idx, 'relative_depth'], test_size=0.3, random_state=42)

# standardize the training data
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)

# apply the same standardization to the testing data
X_test_std = scaler.transform(X_test)

In [ ]:
X_train_std

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Dense
import numpy as np
import matplotlib.pyplot as plt

# Define the neural network architecture
model = Sequential()
model.add(Dense(, input_dim=3, activation='relu'))
model.add(Dense(64))
model.add(Dense(1))

# Compile the model
model.compile(loss='mean_squared_error', optimizer='adam')

# Train the model and plot the learning loss
history = model.fit(X_train, y_train, epochs=100, batch_size=512, validation_data=(X_test, y_test), verbose=0)

In [ ]:
plt.plot(history.history['loss'], label='training')
plt.plot(history.history['val_loss'], label='validation')
plt.title('Learning Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# kernel regression

In [ ]:
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error
import numpy as np
import matplotlib.pyplot as plt

# Define the parameter grid for hyperparameter tuning
param_grid = {'alpha': np.logspace(-3, 3, 7),
              'kernel': ['linear', 'rbf', 'poly']}

# Define the cross-validation strategy
cv = KFold(n_splits=10)

# Create a kernel ridge regression model
clf = KernelRidge()

# Perform grid search cross-validation to select hyperparameters
grid_search = GridSearchCV(clf, param_grid=param_grid, cv=cv, scoring='neg_mean_squared_error')
grid_search.fit(X_train_std, y_train)

# Get the best hyperparameters
best_params = grid_search.best_params_

# Create a new kernel ridge regression model with the best hyperparameters
clf = KernelRidge(alpha=best_params['alpha'], kernel=best_params['kernel'])

# Fit the model to the data
clf.fit(X_train_std, y_train)

# Predict on new data
# y_pred = clf.predict(X)

# Compute the mean squared error on the test set
mse = mean_squared_error(y_train, clf.predict(X_train_std))

In [ ]:
# Plot the mean squared error as a function of alpha
alphas = np.logspace(-3, 3, 7)
mse_values = -grid_search.cv_results_['mean_test_score']
plt.plot(alphas, mse_values)
plt.xscale('log')
plt.xlabel('Alpha')
plt.ylabel('MSE')
plt.show()

In [ ]:
mse_values = -grid_search.cv_results_['mean_test_score']

In [ ]:
alphas 

In [ ]:
mse_values

In [ ]:
best_params

In [ ]:
clf.

# save and load

In [ ]:
filename = 'rrbs_model.sav'
pickle.dump(clf, open(filename, 'wb'))
 
# some time later...
 
# load the model from disk
clf = pickle.load(open(filename, 'rb'))
relative_score = clf.predict(X_new)

In [ ]:
np.sum(frag_df.flag_start + frag_df.flag_end != 0)

# Misc

In [ ]:
is_upstream = False
pos = 182849
contig_id ='chr1'

In [ ]:
find_closest_cut(contig_id, pos, site_list, is_upstream = is_upstream)

In [ ]:
pos + pos_arr[np.argmin(np.abs(pos_arr))]

In [ ]:
ref_dict[contig_id][51586:51586]

In [ ]:
find_closest_cut('chr1',51587, site_list, is_upstream=True)

In [ ]:
read_gen = read_pair_generator(bam_file, 'chr21')

In [ ]:
read1, read2 = next(read_gen)

In [ ]:
read1.reference_start

In [ ]:
read1.reference_end

In [ ]:
read2.reference_start

In [ ]:
read2.reference_end

In [ ]:
read1.mapq

In [ ]:
read1.cigar

In [ ]:
read1.query_name

In [ ]:
read2.query_name

In [ ]:
read1.aligned_pairs

In [ ]:
bam = pysam.AlignmentFile(bam_file, "rb")

for _, reads in itertools.groupby(bam.fetch(), key=lambda x: x.query_name):
    reads = list(reads)
    if len(reads) == 2:
        read1, read2 = reads
        if read1.is_read1 and read2.is_read2 and read1.reference_name == read2.reference_name:
            print("Pair:", read1.query_name)
            print("Read 1:", read1.query_name, read1.reference_start, read1.reference_end)
            print("Read 2:", read2.query_name, read2.reference_start, read2.reference_end)
            break

In [ ]:
read1.aend

In [ ]:
read1.reference_end

In [ ]:
read1.cigarstring

In [ ]:
read1.reference_start

In [ ]:
read1.aligned_pairs

In [ ]:
reads[0].seq

In [ ]:
reads[0].reference_name

In [ ]:
ref_dict[reads[0].reference_name][reads[0].reference_start:reads[0].reference_end]

In [ ]:
ref_dict[reads[1].reference_name][reads[1].reference_start:reads[1].reference_end]

In [ ]:
print(reads[0].query_name)